<a href="https://colab.research.google.com/github/titoug06-commits/MYOTON2026/blob/main/In_Vivo_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Volet In Vivo

Ce notebook reproduit l'ensemble des analyses statistiques et des
figures présentées dans la partie **Résultats Volet In Vivo** du
mémoire.

**Fichiers requis** (à placer dans le même dossier que ce notebook) :
- `Myoton.csv` : mesures brutes du MyotonPRO
- `SWE.csv` : mesures d'élastographie par ondes de cisaillement
- `THICK.csv` : épaisseurs mesurées par analyse d'image échographique

Les figures sont sauvegardées dans le dossier `Figures/`.

Le notebook suit l'ordre des résultats présentés dans le mémoire :
1. Import des librairies et préparation des données
2. Fiabilité des mesures (ICC, 3 vs 5 impulsions)
3. Analyse descriptive (boxplots)
4. Association raideur MyotonPRO / raideur SWE
5. Prédicteurs de la raideur MyotonPRO (régression pas-à-pas, LME, PCR)
6. Effet confondant de l'épaisseur de gras
7. Proportion de fascia et capacité du MyotonPRO à refléter le fascia
8. Annexe : analyse complémentaire non retenue dans le corps du mémoire


## 1. Import des librairies et configuration

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings('ignore')

# Dossier de sortie pour les figures
os.makedirs('Figures', exist_ok=True)

# Style graphique commun à toutes les figures du mémoire
plt.rcParams.update({
    'font.family':        'serif',
    'font.size':           11,
    'axes.titlesize':      12,
    'axes.labelsize':      11,
    'legend.fontsize':     10,
    'xtick.labelsize':     10,
    'ytick.labelsize':     10,
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'figure.dpi':          200,
    'savefig.dpi':         200,
    'savefig.bbox':        'tight',
    'savefig.pad_inches':  0.1,
})

COULEURS = {
    'décubitus': '#4393C3',   # bleu
    'assis':     '#D6604D',   # rouge-orangé
}
ALPHA_SCATTER = 0.55


## 2. Préparation des données

Chargement et nettoyage des trois fichiers sources, puis fusion en
une base de données unique (`DF`), utilisée par l'ensemble des
analyses qui suivent.

In [ ]:
def clean_num(x):
    """Convertit une valeur texte (virgule décimale) en flottant."""
    return pd.to_numeric(str(x).replace(',', '.'), errors='coerce')


# Correspondance entre le nom des participants et leur code anonymisé
mapping_noms_codes = {
    "Juliette Siberchicot": "JS03", "Marie-Margot Joaness": "MMJ04", "Vincent Rozzi": "VR05",
    "Guy Fructus": "GF06", "Alice Bredel": "AB07", "Andrea Deville": "AD08",
    "Combis Lucas": "CL09", "Gaspard Gouin": "GG10", "Hugo Tarsitano Renzetti": "HT11",
    "Lise Claveirole": "LC12", "Perez Maxence": "MP13", "Balthazar Soulier": "BS14",
    "Vera Lowe": "VL15", "Myriam Levite": "ML16", "Alkeos Michos-Noury": "AMN17",
    "Marie Erceau": "ME18", "Anselme Ahmed omar": "AO19", "Chloe Briard": "CB20",
    "Louane Prudhon": "LP21", "Loan Fonteneau": "LF22", "Antoine Piraudon": "AP23",
    "Constant Lheritier": "CL24", "Vincent Noirot": "VN25", "Thomas Constantin": "TC26",
    "Anis Belmimoun": "AB27", "Maxence Lamouille": "ML28", "Adrien Velferinger": "AV29",
}

CLES = ['Code participants', 'Localisation', 'Côté', 'Position']


def load_myoton_tlb(path='Myoton.csv'):
    """Charge et nettoie les mesures MyotonPRO brutes (une ligne par
    impulsion). Retourne un DataFrame non moyenné, réutilisé à la
    fois pour construire la base fusionnée (DF) et pour le calcul de
    fiabilité (ICC), qui a besoin des mesures individuelles.
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    # Filtrage sur le site lombaire (TLB), exclusion des séries à 3
    # impulsions et des sites non lombaires (Plantar/Foot)
    df = df[df['Pattern'].str.contains('TLB', na=False)].copy()
    df = df[~df['Pattern'].str.contains(r'\(3\)|Plantar|Foot', case=False, na=False)]

    # Normalisation des variables catégorielles à partir du Pattern
    df['Localisation'] = df['Pattern'].apply(
        lambda x: 'L4/L5' if 'L4-5' in str(x) else 'L2/L3')
    df['Côté'] = df['Side'].str.strip().replace({'Left': 'G', 'Right': 'D'})
    df['Position'] = df['Pattern'].apply(
        lambda p: 'assis' if 'sit' in str(p).lower() else 'décubitus')

    # Nettoyage numérique et anonymisation
    df['Stiffness'] = df['Stiffness'].apply(clean_num)
    df['Subject name'] = df['Subject name'].apply(lambda x: " ".join(str(x).split()))
    df['Code participants'] = df['Subject name'].map(mapping_noms_codes)
    df = df.dropna(subset=['Code participants', 'Stiffness'])

    return df


def process_data_final():
    """Construit la base de données fusionnée (SWE + épaisseurs +
    MyotonPRO moyenné par configuration participant x site x côté x
    position), utilisée par l'ensemble des analyses du mémoire.
    """
    df_myo = load_myoton_tlb('Myoton.csv')
    df_myo_avg = (df_myo.groupby(CLES)['Stiffness'].mean()
                  .reset_index().rename(columns={'Stiffness': 'Myoton_Stiffness'}))

    df_swe = pd.read_csv('SWE.csv')
    df_swe.columns = df_swe.columns.str.strip()
    df_swe['SWE_Young_Mean'] = df_swe['SWE Young Mean'].apply(clean_num)
    df_swe['Position'] = df_swe['Position'].str.strip().str.lower()
    df_swe['Côté'] = df_swe['Côté'].str.strip()

    df_thick = pd.read_csv('THICK.csv')
    df_thick.columns = df_thick.columns.str.strip()
    cols_thick = ['Mean Thick Epi (mm)', 'Mean Thick petit gras (mm)',
                  'Mean Thick TL (mm)', 'Mean GRAS (mm)', 'Mean Complexe FTL (mm)']
    for col in cols_thick:
        df_thick[col] = df_thick[col].apply(clean_num)
    df_thick['Position'] = df_thick['Position'].str.strip().str.lower()
    df_thick['Côté'] = df_thick['Côté'].str.strip()

    # Fusion SWE + épaisseurs, puis ajout de la raideur MyotonPRO
    merged_st = pd.merge(df_swe[CLES + ['SWE_Young_Mean']],
                          df_thick[CLES + cols_thick], on=CLES, how='inner')
    final_df = pd.merge(merged_st, df_myo_avg, on=CLES, how='inner')

    return final_df.drop_duplicates()


DF = process_data_final()

# Variables muettes utilisées par la PCR et les cercles de corrélation
DF['Position_assis'] = (DF['Position'] == 'assis').astype(int)
DF['Localisation_L2/L3'] = (DF['Localisation'] == 'L2/L3').astype(int)

print(f"Base de données finale : {len(DF)} observations, "
      f"{DF['Code participants'].nunique()} participants.")
DF.head()


Base de données finale : 216 observations, 27 participants.


,Code participants,Localisation,Côté,Position,SWE_Young_Mean,Mean Thick Epi (mm),Mean Thick petit gras (mm),Mean Thick TL (mm),Mean GRAS (mm),Mean Complexe FTL (mm),Myoton_Stiffness,Position_assis,Localisation_L2/L3
0,JS03,L4/L5,D,décubitus,46.766667,0.374222,0.318111,2.015444,5.565111,2.707778,189.2,0,0
1,JS03,L4/L5,G,décubitus,53.566667,0.318667,0.427778,2.388444,5.078444,3.134889,197.2,0,0
2,JS03,L2/L3,D,décubitus,45.433333,0.470333,0.259889,1.891000,4.338667,2.621222,334.8,0,1
3,JS03,L2/L3,G,décubitus,56.766667,0.530778,0.348111,1.832333,4.181111,2.711222,324.5,0,1
4,JS03,L4/L5,D,assis,143.933333,0.399333,0.342556,1.622111,4.475889,2.364000,320.7,1,0


## 3. Fiabilité des mesures MyotonPRO (section 3.1 du mémoire)

### 3.1 Coefficient de corrélation intraclasse (ICC)

In [ ]:
def compute_icc(df_myo):
    """Calcule l'ICC(3,1) et l'ICC(3,k) à partir des mesures MyotonPRO
    individuelles (non moyennées), en ne conservant que les
    configurations disposant exactement de 10 mesures unitaires (deux
    séries de 5 impulsions consécutives).
    """
    df = df_myo.copy()
    df['subject_id'] = (df['Code participants'] + '_' + df['Localisation']
                         + '_' + df['Côté'] + '_' + df['Position'])

    group_counts = df.groupby('subject_id').size()
    valid_groups = group_counts[group_counts == 10].index
    df_icc = df[df['subject_id'].isin(valid_groups)].copy()

    # Tableau configurations (lignes) x répétitions (colonnes)
    data_pivot = df_icc.pivot_table(
        index='subject_id',
        columns=df_icc.groupby('subject_id').cumcount(),
        values='Stiffness')

    n = data_pivot.shape[0]  # nombre de configurations
    k = data_pivot.shape[1]  # nombre de répétitions (10 = 2 x 5 impulsions)

    # ANOVA à deux facteurs (modèle 3, effets mixtes)
    grand_mean = data_pivot.values.mean()
    SST = ((data_pivot.values - grand_mean) ** 2).sum()
    SSB = k * ((data_pivot.mean(axis=1) - grand_mean) ** 2).sum()
    SSW = ((data_pivot.values.T - data_pivot.mean(axis=1).values) ** 2).sum()
    SSJ = n * ((data_pivot.mean(axis=0) - grand_mean) ** 2).sum()
    SSE = SSW - SSJ

    MSB = SSB / (n - 1)
    MSE = SSE / ((n - 1) * (k - 1))

    icc_31 = (MSB - MSE) / (MSB + (k - 1) * MSE)
    icc_3k = (MSB - MSE) / MSB

    return n, k, icc_31, icc_3k


df_myo_raw = load_myoton_tlb('Myoton.csv')
n_config, k_rep, icc_31, icc_3k = compute_icc(df_myo_raw)

print(f"Nombre de configurations analysées : {n_config}")
print(f"Nombre de répétitions par configuration (2 x 5 impulsions) : {k_rep}")
print(f"ICC(3,1) (mesure unique)      : {icc_31:.3f}")
print(f"ICC(3,k) (moyenne des {k_rep})       : {icc_3k:.3f}")


Nombre de configurations analysées : 216
Nombre de répétitions par configuration (2 x 5 impulsions) : 10
ICC(3,1) (mesure unique)      : 0.985
ICC(3,k) (moyenne des 10)       : 0.998


### 3.2 Concordance entre 3 et 5 impulsions

In [ ]:
def process_data_3v5(df_myo_raw_path='Myoton.csv'):
    """Prépare la comparaison entre le protocole standard (5 impulsions)
    et un protocole allégé (3 impulsions), disponible pour un
    sous-ensemble de participants (7).
    """
    df = pd.read_csv(df_myo_raw_path)
    df.columns = df.columns.str.strip()

    df['Is_3_Imp'] = df['Pattern'].str.contains(r'\(3\)', na=False)
    df['Localisation'] = df['Pattern'].apply(
        lambda x: 'L4/L5' if 'L4-5' in str(x)
        else ('L2/L3' if 'L2-3' in str(x) else 'Plantar'))
    df['Côté'] = df['Side'].str.strip().replace({'Left': 'G', 'Right': 'D'})
    df['Position'] = df['Pattern'].apply(
        lambda p: 'assis' if 'sit' in str(p).lower() else 'décubitus')
    df['Stiffness'] = df['Stiffness'].apply(clean_num)

    df['Subject name clean'] = df['Subject name'].apply(lambda x: " ".join(str(x).split()))
    df['Code participants'] = df['Subject name clean'].map(mapping_noms_codes)

    # Participants ayant bénéficié des deux protocoles (3 et 5 impulsions)
    codes_7 = ['AV29', 'ML28', 'AB27', 'TC26', 'VN25', 'CL24', 'AP23']
    df = df[df['Code participants'].isin(codes_7)].dropna(subset=['Stiffness'])

    df_avg = df.groupby(CLES + ['Is_3_Imp'])['Stiffness'].mean().reset_index()
    df_5 = df_avg[~df_avg['Is_3_Imp']].rename(columns={'Stiffness': 'Stiffness_5'})
    df_3 = df_avg[df_avg['Is_3_Imp']].rename(columns={'Stiffness': 'Stiffness_3'})

    return pd.merge(df_5[CLES + ['Stiffness_5']], df_3[CLES + ['Stiffness_3']], on=CLES)


def analyze_concordance(df_comp):
    """Calcule l'écart relatif et l'ICC(3,1) de concordance entre les
    mesures à 3 et à 5 impulsions.
    """
    n = len(df_comp)
    df_comp = df_comp.copy()
    df_comp['Diff_Pct'] = (abs(df_comp['Stiffness_3'] - df_comp['Stiffness_5'])
                            / df_comp['Stiffness_5']) * 100

    k = 2  # deux méthodes comparées (3 vs 5 impulsions)
    values = df_comp[['Stiffness_5', 'Stiffness_3']].values
    grand_mean = values.mean()

    ss_between = k * ((values.mean(axis=1) - grand_mean) ** 2).sum()
    ss_within = ((values.T - values.mean(axis=1)) ** 2).sum()
    ss_raters = n * ((values.mean(axis=0) - grand_mean) ** 2).sum()
    ss_error = ss_within - ss_raters

    ms_between = ss_between / (n - 1)
    ms_error = ss_error / ((n - 1) * (k - 1))
    icc_31_conc = (ms_between - ms_error) / (ms_between + (k - 1) * ms_error)

    print(f"Nombre de comparaisons directes : {n}")
    print(f"Écart relatif moyen : {df_comp['Diff_Pct'].mean():.2f} %")
    print(f"Écart relatif max   : {df_comp['Diff_Pct'].max():.2f} %")
    print(f"ICC(3,1) de concordance (3 vs 5 impulsions) : {icc_31_conc:.4f}")

    return df_comp


def fig_3vs5(df_comp, outpath='Figures/3VS5.png'):
    """Figure 3.1 (mémoire) : concordance entre 3 et 5 impulsions."""
    x, y = df_comp['Stiffness_5'], df_comp['Stiffness_3']
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    r2 = r_value ** 2

    fig, ax = plt.subplots(Figuresize=(5, 5))
    ax.plot(x, y, 'o', color='k', markerfacecolor='none', markersize=6,
            label=f"mesures Myoton r\u00b2={r2:.3f}")
    lim = [min(x.min(), y.min()) * 0.95, max(x.max(), y.max()) * 1.05]
    ax.plot(lim, lim, 'k-', label='$y = x$')
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel(r'Raideur Myoton — 5 impulsions (N$\cdot$m$^{-1}$)')
    ax.set_ylabel(r'Raideur Myoton — 3 impulsions (N$\cdot$m$^{-1}$)')
    ax.set_title('Raideur 3 vs 5 impulsions')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(outpath)
    print(f'Saved: {outpath}')
    plt.close(fig)


df_comparaison = process_data_3v5('Myoton.csv')
df_comparaison = analyze_concordance(df_comparaison)
fig_3vs5(df_comparaison)


Nombre de comparaisons directes : 70
Écart relatif moyen : 5.59 %
Écart relatif max   : 33.03 %
ICC(3,1) de concordance (3 vs 5 impulsions) : 0.9756
Saved: Figures/3VS5.png


## 4. Analyse descriptive (section 3.2 du mémoire)

In [ ]:
def fig_boxplots(df, outpath='Figures/boxplots.png'):
    """Figure 3.2 (mémoire) : distribution de la raideur MyotonPRO par
    position, localisation et côté.
    """
    fig, axes = plt.subplots(1, 3, Figuresize=(12, 4))

    panel_specs = [
        ('Position',     ['décubitus', 'assis'], 'Position'),
        ('Localisation', ['L4/L5', 'L2/L3'],     'Localisation'),
        ('Côté',         ['D', 'G'],              'Côté'),
    ]

    for ax, (col, order, xlabel) in zip(axes, panel_specs):
        data = [df.loc[df[col] == g, 'Myoton_Stiffness'].dropna() for g in order]
        bp = ax.boxplot(
            data, labels=order, patch_artist=True,
            medianprops=dict(color='black', lw=1.5),
            meanprops=dict(linestyle='--', color='gray', lw=1),
            showmeans=True, meanline=True, showfliers=False,
        )
        colors = ['#4393C3', '#D6604D']
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(r'Raideur Myoton (N$\cdot$m$^{-1}$)' if ax is axes[0] else '')
        mean_line = Line2D([0], [0], color='gray', linestyle='--', lw=1)
        ax.legend([mean_line], ['Moyenne'], loc='upper right', frameon=False)

    fig.tight_layout()
    fig.savefig(outpath)
    print(f'Saved: {outpath}')
    plt.close(fig)


fig_boxplots(DF)


Saved: Figures/boxplots.png


## 5. Association raideur MyotonPRO / raideur SWE (section 3.3 du mémoire)

In [ ]:
def fig_correlation(df, outpath='Figures/corr.png'):
    """Figure 3.3 (mémoire) : association entre raideur MyotonPro et
    épaisseur de gras (panneau gauche), et entre raideur MyotonPro et
    module de Young SWE, par position (panneau droit).
    """
    fig, axes = plt.subplots(1, 2, Figuresize=(10, 4.5))

    # Panneau gauche : Myoton ~ GRAS
    ax = axes[0]
    for pos, col in COULEURS.items():
        sub = df[df['Position'] == pos].dropna(subset=['Mean GRAS (mm)', 'Myoton_Stiffness'])
        sl, ic, r, p, _ = stats.linregress(sub['Mean GRAS (mm)'], sub['Myoton_Stiffness'])
        xs = np.linspace(sub['Mean GRAS (mm)'].min(), sub['Mean GRAS (mm)'].max(), 100)
        ptext = 'p<0,001' if p < 0.001 else f'p={p:.3f}'
        ax.scatter(sub['Mean GRAS (mm)'], sub['Myoton_Stiffness'],
                   color=col, alpha=ALPHA_SCATTER, s=25, label=f'{pos} (r={r:.2f}, {ptext})')
        ax.plot(xs, sl * xs + ic, color=col, linestyle='--', lw=1.5)
    ax.set_xlabel('Épaisseur de tissu adipeux — GRAS (mm)')
    ax.set_ylabel(r'Raideur Myoton (N$\cdot$m$^{-1}$)')
    ax.set_title("Myoton en fonction de l'épaisseur de gras")
    ax.legend(frameon=False)

    # Panneau droit : Myoton ~ SWE
    ax = axes[1]
    for pos, col in COULEURS.items():
        sub_pos = df[df['Position'] == pos].dropna(subset=['SWE_Young_Mean', 'Myoton_Stiffness'])
        sl, ic, r, p, _ = stats.linregress(sub_pos['SWE_Young_Mean'], sub_pos['Myoton_Stiffness'])
        xs = np.linspace(df['SWE_Young_Mean'].min(), df['SWE_Young_Mean'].max(), 100)
        ptext = 'p<0,001' if p < 0.001 else f'p={p:.3f}'
        ax.scatter(sub_pos['SWE_Young_Mean'], sub_pos['Myoton_Stiffness'],
                   color=col, marker='o', alpha=ALPHA_SCATTER, s=25,
                   label=f'{pos} (r={r:.2f}, {ptext})')
        ax.plot(xs, sl * xs + ic, color=col, linestyle='--', lw=1.5)
        print(f"Corrélation SWE-Myoton, position {pos} : r={r:.3f}, p={p:.4g}")
    ax.set_xlabel('Module de Young SWE (kPa)')
    ax.set_ylabel('')
    ax.set_title('Corrélation SWE — Myoton')
    ax.legend(frameon=False, fontsize=9)

    fig.tight_layout()
    fig.savefig(outpath)
    print(f'Saved: {outpath}')
    plt.close(fig)


fig_correlation(DF)


Corrélation SWE-Myoton, position décubitus : r=0.502, p=3.2e-08
Corrélation SWE-Myoton, position assis : r=0.520, p=8.186e-09
Saved: Figures/corr.png


## 6. Prédicteurs de la raideur MyotonPRO (section 3.4 du mémoire)

### 6.1 Régression pas-à-pas (forward et backward, critère AIC)

In [ ]:
def stepwise_forward(df, dependent_variable, initial_predictors):
    """Sélection ascendante (forward) par critère AIC."""
    selected, best_aic = [], float('inf')
    print("### Régression pas-à-pas : sélection forward ###")

    while True:
        candidate, candidate_aic = None, best_aic
        for p in [p for p in initial_predictors if p not in selected]:
            formula = f"{dependent_variable} ~ {' + '.join(selected + [p])}"
            aic = smf.ols(formula=formula, data=df).fit().aic
            if aic < candidate_aic:
                candidate_aic, candidate = aic, p
        if candidate is None:
            break
        selected.append(candidate)
        best_aic = candidate_aic
        print(f"Ajouté : {candidate} (nouvel AIC : {best_aic:.2f})")

    formula = f"{dependent_variable} ~ {' + '.join(selected) if selected else '1'}"
    results = smf.ols(formula=formula, data=df).fit()
    print("\n--- Modèle final (forward) ---")
    print(results.summary())
    return results


def stepwise_backward(df, dependent_variable, initial_predictors):
    """Élimination descendante (backward) par critère AIC."""
    selected = list(initial_predictors)
    formula = f"{dependent_variable} ~ {' + '.join(selected)}"
    best_aic = smf.ols(formula=formula, data=df).fit().aic
    print("\n### Régression pas-à-pas : élimination backward ###")
    print(f"Modèle initial avec tous les prédicteurs (AIC : {best_aic:.2f})")

    while True:
        worst, best_aic_after_removal = None, best_aic
        for p in selected:
            remaining = [x for x in selected if x != p]
            formula = f"{dependent_variable} ~ {' + '.join(remaining) if remaining else '1'}"
            aic = smf.ols(formula=formula, data=df).fit().aic
            if aic < best_aic_after_removal:
                best_aic_after_removal, worst = aic, p
        if worst is None:
            break
        selected.remove(worst)
        best_aic = best_aic_after_removal
        print(f"Retiré : {worst} (nouvel AIC : {best_aic:.2f})")

    formula = f"{dependent_variable} ~ {' + '.join(selected) if selected else '1'}"
    results = smf.ols(formula=formula, data=df).fit()
    print("\n--- Modèle final (backward) ---")
    print(results.summary())
    return results


dependent_variable = 'Myoton_Stiffness'
initial_potential_predictors = [
    "Q('SWE_Young_Mean')",
    "Q('Mean GRAS (mm)')",
    "Q('Mean Complexe FTL (mm)')",
    "C(Localisation)",
    "C(Position)",
]

results_forward = stepwise_forward(DF, dependent_variable, initial_potential_predictors)
results_backward = stepwise_backward(DF, dependent_variable, initial_potential_predictors)


### Régression pas-à-pas : sélection forward ###
Ajouté : Q('SWE_Young_Mean') (nouvel AIC : 2840.33)
Ajouté : Q('Mean GRAS (mm)') (nouvel AIC : 2797.52)
Ajouté : C(Position) (nouvel AIC : 2750.49)
Ajouté : C(Localisation) (nouvel AIC : 2747.48)

--- Modèle final (forward) ---
                            OLS Regression Results                            
Dep. Variable:       Myoton_Stiffness   R-squared:                       0.732
Model:                            OLS   Adj. R-squared:                  0.727
Method:                 Least Squares   F-statistic:                     144.2
Date:                Wed, 19 Aug 2026   Prob (F-statistic):           3.39e-59
Time:                        12:27:32   Log-Likelihood:                -1368.7
No. Observations:                 216   AIC:                             2747.
Df Residuals:                     211   BIC:                             2764.
Df Model:                           4                                         
Covariance T

### 6.2 Modèle linéaire à effets mixtes (LME)

In [ ]:
fixed_effects_formula = "Myoton_Stiffness ~ Q('SWE_Young_Mean') + Q('Mean GRAS (mm)') + C(Position) + C(Localisation)"

lme_model = smf.mixedlm(fixed_effects_formula, data=DF, groups=DF['Code participants'])
lme_result = lme_model.fit()

print(lme_result.summary())


                  Mixed Linear Model Regression Results
Model:                 MixedLM    Dependent Variable:    Myoton_Stiffness
No. Observations:      216        Method:                REML            
No. Groups:            27         Scale:                 16077.6747      
Min. group size:       8          Log-Likelihood:        -1350.3338      
Max. group size:       8          Converged:             Yes             
Mean group size:       8.0                                               
-------------------------------------------------------------------------
                          Coef.   Std.Err.   z    P>|z|  [0.025   0.975] 
-------------------------------------------------------------------------
Intercept                 604.861   46.836 12.914 0.000  513.063  696.658
C(Position)[T.décubitus] -225.795   28.731 -7.859 0.000 -282.107 -169.483
C(Localisation)[T.L4/L5]   39.257   18.033  2.177 0.029    3.913   74.601
Q('SWE_Young_Mean')         0.910    0.205  4.444 0.000 

### 6.3 Régression sur composantes principales (PCR)

In [ ]:
features = ['SWE_Young_Mean', 'Mean GRAS (mm)', 'Mean Complexe FTL (mm)', 'Position', 'Localisation']

y = DF['Myoton_Stiffness']
X_raw = DF[features].copy()
X = pd.get_dummies(X_raw, columns=['Position', 'Localisation'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Analyse en composantes principales (ajustée sur le jeu d'entraînement)
pca = PCA(n_components=None)
pca.fit(X_train_scaled)

print("Variance expliquée par chaque composante principale :")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i + 1}: {ratio:.3f}")
print(f"Variance cumulée expliquée : {np.cumsum(pca.explained_variance_ratio_)}")

X_train_pca = pca.transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Régression linéaire sur les composantes principales
pcr_model = LinearRegression()
pcr_model.fit(X_train_pca, y_train)
y_pred_pcr = pcr_model.predict(X_test_pca)

r2_pcr = r2_score(y_test, y_pred_pcr)
mse_pcr = mean_squared_error(y_test, y_pred_pcr)
rmse_pcr = np.sqrt(mse_pcr)

print(f"\nPCR — R² (jeu de test)   : {r2_pcr:.3f}")
print(f"PCR — MSE (jeu de test)  : {mse_pcr:.3f}")
print(f"PCR — RMSE (jeu de test) : {rmse_pcr:.3f}")

print("\nCoefficients du modèle PCR (sur les composantes principales) :")
for i, coef in enumerate(pcr_model.coef_):
    print(f"  PC{i + 1}: {coef:.3f}")
print(f"  Intercept: {pcr_model.intercept_:.3f}")

original_feature_names = X.columns.tolist()
print("\nContribution de chaque variable à chaque composante :")
for i, pc_loadings in enumerate(pca.components_):
    print(f"\nPC{i + 1}:")
    for feature, loading in zip(original_feature_names, pc_loadings):
        print(f"  {feature}: {loading:.3f}")


Variance expliquée par chaque composante principale :
  PC1: 0.427
  PC2: 0.233
  PC3: 0.209
  PC4: 0.087
  PC5: 0.043
Variance cumulée expliquée : [0.42653078 0.65988649 0.86934936 0.95663179 1.        ]

PCR — R² (jeu de test)   : 0.762
PCR — MSE (jeu de test)  : 18609.141
PCR — RMSE (jeu de test) : 136.415

Coefficients du modèle PCR (sur les composantes principales) :
  PC1: 145.028
  PC2: -32.851
  PC3: -17.603
  PC4: -71.176
  PC5: -38.643
  Intercept: 499.735

Contribution de chaque variable à chaque composante :

PC1:
  SWE_Young_Mean: 0.605
  Mean GRAS (mm): -0.469
  Mean Complexe FTL (mm): -0.417
  Position_décubitus: -0.490
  Localisation_L4/L5: 0.009

PC2:
  SWE_Young_Mean: -0.270
  Mean GRAS (mm): -0.523
  Mean Complexe FTL (mm): -0.450
  Position_décubitus: 0.543
  Localisation_L4/L5: -0.396

PC3:
  SWE_Young_Mean: -0.122
  Mean GRAS (mm): 0.028
  Mean Complexe FTL (mm): -0.448
  Position_décubitus: 0.220
  Localisation_L4/L5: 0.857

PC4:
  SWE_Young_Mean: -0.016
  Mean G

### 6.4 Cercles de corrélation des variables (ACP)

In [ ]:
def fig_pca_circle(df, pc_x=0, pc_y=1, outpath='Figures/PC1VSPC2.png'):
    """Cercle de corrélation des variables pour l'ACP, sur le plan
    (PC{pc_x+1}, PC{pc_y+1}). Fonction utilisée pour les
    trois plans présentés dans le mémoire (PC1-PC2, PC1-PC3) et en
    annexe (PC2-PC3).
    """
    feats = ['SWE_Young_Mean', 'Mean GRAS (mm)', 'Mean Complexe FTL (mm)',
             'Position_assis', 'Localisation_L2/L3']
    labels = ['SWE Young Mean', 'GRAS (mm)', 'FTL (mm)',
              'Position assis', 'Localisation L2/L3']

    X = df[feats].dropna()
    Xs = StandardScaler().fit_transform(X)
    pca = PCA(n_components=5).fit(Xs)
    var = pca.explained_variance_ratio_

    # Corrélation variable–composante (loadings normalisés)
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    lx, ly = loadings[:, pc_x], loadings[:, pc_y]

    fig, ax = plt.subplots(Figuresize=(6, 6))
    ax.add_patch(plt.Circle((0, 0), 1, color='gray', linestyle='--', fill=False, alpha=0.5))
    for i, (x, y) in enumerate(zip(lx, ly)):
        ax.annotate('', xy=(x, y), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
        ha = 'left' if x >= 0 else 'right'
        offset = 0.05 if x >= 0 else -0.05
        ax.text(x + offset, y, labels[i], fontsize=9, color='k', ha=ha, va='center')

    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.axvline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel(f'PC{pc_x + 1} ({var[pc_x] * 100:.1f} % de variance)')
    ax.set_ylabel(f'PC{pc_y + 1} ({var[pc_y] * 100:.1f} % de variance)')
    ax.set_aspect('equal')
    fig.tight_layout()
    fig.savefig(outpath)
    print(f'Saved: {outpath}')
    plt.close(fig)


# Figure 3.4 du mémoire (PC1 vs PC2)
fig_pca_circle(DF, pc_x=0, pc_y=1, outpath='Figures/PC1VSPC2.png')
# Figure 3.5 du mémoire (PC1 vs PC3)
fig_pca_circle(DF, pc_x=0, pc_y=2, outpath='Figures/PC1VSPC3.png')
# Plan complémentaire, cité en annexe (PC2 vs PC3)
fig_pca_circle(DF, pc_x=1, pc_y=2, outpath='Figures/PC2VSPC3.png')


Saved: Figures/PC1VSPC2.png
Saved: Figures/PC1VSPC3.png
Saved: Figures/PC2VSPC3.png


## 7. Effet confondant de l'épaisseur de gras (section 3.5 du mémoire)


In [ ]:
def fig_tertiles_gras(df, seuil_bas=1, seuil_haut=4, outpath='Figures/groupes_gras.png'):
    """Figure 3.6 (mémoire) : régression Myoton ~ SWE pour deux groupes
    contrastés d'épaisseur de tissu adipeux (gras faible / gras élevé).
    """
    df2 = df.dropna(subset=['Mean GRAS (mm)', 'SWE_Young_Mean', 'Myoton_Stiffness']).copy()

    label_bas = f'Gras faible (< {seuil_bas}mm)'
    label_haut = f'Gras élevé (> {seuil_haut}mm)'
    df2['groupe_gras'] = pd.Series(np.nan, index=df2.index, dtype=object)
    df2.loc[df2['Mean GRAS (mm)'] < seuil_bas, 'groupe_gras'] = label_bas
    df2.loc[df2['Mean GRAS (mm)'] > seuil_haut, 'groupe_gras'] = label_haut
    df2 = df2.dropna(subset=['groupe_gras'])

    couleurs_t = {label_bas: '#2166AC', label_haut: '#D6604D'}

    fig, ax = plt.subplots(Figuresize=(6, 5))
    for label, col in couleurs_t.items():
        sub = df2[df2['groupe_gras'] == label]
        ax.scatter(sub['SWE_Young_Mean'], sub['Myoton_Stiffness'],
                   color=col, alpha=ALPHA_SCATTER, s=22)
        sl, ic, r, p, _ = stats.linregress(sub['SWE_Young_Mean'], sub['Myoton_Stiffness'])
        xs = np.linspace(df2['SWE_Young_Mean'].min(), df2['SWE_Young_Mean'].max(), 100)
        ptext = 'p<0,001' if p < 0.001 else f'p={p:.3f}'
        ax.plot(xs, sl * xs + ic, color=col, lw=2, label=f'{label}  (r={r:.2f}, {ptext})')

    ax.set_xlabel('Module de Young SWE (kPa)')
    ax.set_ylabel(r'Raideur Myoton (N$\cdot$m$^{-1}$)')
    ax.set_title(f'Régression Myoton ~ SWE par groupes de GRAS < {seuil_bas}mm et > {seuil_haut}mm')
    ax.legend(frameon=False, loc='upper left')
    fig.tight_layout()
    fig.savefig(outpath)
    print(f'Saved: {outpath}')
    plt.close(fig)


fig_tertiles_gras(DF)


Saved: Figures/groupes_gras.png


## 8. Proportion de fascia et capacité du MyotonPRO à refléter le fascia (section 3.6 du mémoire)

In [ ]:
def fig_proportion_fascia(df, outpath='Figures/proportion_fascia.png'):
    """Figure 3.7 (mémoire) : raideur MyotonPRO en fonction de la
    proportion de fascia superficiel dans l'épaisseur totale.
    """
    df2 = df.dropna(subset=['Mean GRAS (mm)', 'Mean Complexe FTL (mm)',
                             'Mean Thick TL (mm)', 'Myoton_Stiffness']).copy()

    denom = df2['Mean GRAS (mm)'] + df2['Mean Complexe FTL (mm)']
    df2['proportion_fascia'] = df2['Mean Thick TL (mm)'] / denom.replace(0, np.nan)
    df2 = df2.dropna(subset=['proportion_fascia'])

    fig, ax = plt.subplots(Figuresize=(6, 5))
    for pos, col in COULEURS.items():
        sub = df2[df2['Position'] == pos].dropna(subset=['proportion_fascia', 'Myoton_Stiffness'])
        if sub.empty:
            continue
        ax.scatter(sub['proportion_fascia'], sub['Myoton_Stiffness'],
                   color=col, alpha=ALPHA_SCATTER, s=25, label=pos)
        sl, ic, r, p, _ = stats.linregress(sub['proportion_fascia'], sub['Myoton_Stiffness'])
        xs = np.linspace(df2['proportion_fascia'].min(), df2['proportion_fascia'].max(), 100)
        ptext = 'p<0,001' if p < 0.001 else f'p={p:.3f}'
        ax.plot(xs, sl * xs + ic, color=col, lw=2, label=f'r={r:.2f} ({pos}), {ptext}')

    ax.set_xlabel('Proportion de fascia : fascia superficiel / (GRAS + complexe fascial)')
    ax.set_ylabel(r'Raideur Myoton (N$\cdot$m$^{-1}$)')
    ax.set_title('Raideur Myoton en fonction de la proportion de fascia superficiel')
    ax.legend(frameon=False, loc='upper left')
    fig.tight_layout()
    fig.savefig(outpath)
    print(f'Saved: {outpath}')
    plt.close(fig)


fig_proportion_fascia(DF)


Saved: Figures/proportion_fascia.png


## 9. Annexe — analyse complémentaire non retenue dans le corps du mémoire

In [ ]:
# Modèle exploratoire testant un terme d'interaction explicite entre
# le module de Young SWE et l'épaisseur de gras. Cette analyse a été
# écartée du corps du mémoire au profit du modèle à effets mixtes
# (LME, section 3.5), plus robuste vis-à-vis de la structure de
# mesures répétées. Elle est conservée ici à titre exploratoire.

modele_ols_interaction = smf.ols(
    formula="Myoton_Stiffness ~ Q('SWE_Young_Mean') + Q('Mean GRAS (mm)') "
            "+ Q('SWE_Young_Mean'):Q('Mean GRAS (mm)')",
    data=DF,
)
resultats_ols_interaction = modele_ols_interaction.fit()
print(resultats_ols_interaction.summary())


                            OLS Regression Results                            
Dep. Variable:       Myoton_Stiffness   R-squared:                       0.700
Model:                            OLS   Adj. R-squared:                  0.696
Method:                 Least Squares   F-statistic:                     165.0
Date:                Wed, 19 Aug 2026   Prob (F-statistic):           3.57e-55
Time:                        12:27:36   Log-Likelihood:                -1381.0
No. Observations:                 216   AIC:                             2770.
Df Residuals:                     212   BIC:                             2783.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------